## First, transform Skyteam_timetable.pdf to doc_0_0.csv

In [1]:
%pip install PyMuPDF matplotlib pdfplumber pandas openpyxl

In [21]:
import os
from concurrent.futures import ProcessPoolExecutor
import pandas as pd
import pymupdf as fitz


def _process_page_chunk(args: tuple[str, int, int]):
    pdf_path, chunk_start, chunk_end = args
    chunk_tables = []

    doc = fitz.open(pdf_path)
    for page_idx in range(chunk_start, chunk_end):
        page = doc[page_idx]
        tabs = page.find_tables()
        for tab in tabs.tables:
            chunk_tables.extend(tab.extract())
            print(f"Отпаршена страница {page_idx + 1}")
    doc.close()

    return chunk_tables


def pdf_to_csv_parallel(
    pdf_path: str,
    output_csv: str,
    start_page: int = 1,
    end_page: int | None = None,
    num_chunks: int = 10,
):
    with fitz.open(pdf_path) as doc:
        total_doc_pages = len(doc)

    start_idx = max(0, start_page - 1)
    end_idx = (
        total_doc_pages
        if end_page is None or end_page > total_doc_pages
        else max(start_idx, end_page)
    )

    if start_idx >= total_doc_pages or start_idx >= end_idx:
        print(f"Некорректный диапазон страниц {start_page} - {end_page}")
        return

    pages_to_process = end_idx - start_idx

    chunks_count = min(num_chunks, pages_to_process)

    chunk_size = pages_to_process // chunks_count
    remainder = pages_to_process % chunks_count

    chunks = []
    curr = start_idx
    for i in range(chunks_count):
        extra = 1 if i < remainder else 0
        nxt = curr + chunk_size + extra
        chunks.append((pdf_path, curr, nxt))
        curr = nxt

    table_data = []

    with ProcessPoolExecutor(max_workers=chunks_count) as executor:
        results = executor.map(_process_page_chunk, chunks)
        for chunk_res in results:
            table_data.extend(chunk_res)

    if not table_data:
        print("Таблиц в файле нет")
        return

    headers = table_data[0]
    max_cols = max(len(headers), max((len(row) for row in table_data[1:]), default=len(headers)))
    
    while len(headers) < max_cols:
        headers.append(f"col_{len(headers) + 1}")

    rows = []
    for row in table_data[1:]:
        if row == headers:
            continue
        if len(row) < max_cols:
            row = row + [""] * (max_cols - len(row))
        elif len(row) > max_cols:
            row = row[:max_cols]
        rows.append(row)

    df = pd.DataFrame(rows, columns=headers)

    # 5. Запись в CSV
    df.to_csv(output_csv, sep="|", index=False, encoding="utf-8-sig")
    print(f"Записано строк: {len(df)}")


if __name__ == "__main__":
    pdf_to_csv_parallel(
        pdf_path="../../resources/raw/Skyteam_Timetable.pdf",
        output_csv="../../resources/interim/Skyteam_Timetable/doc_0_0.csv",
        start_page=5,
        end_page=27514,
        num_chunks=10,  # Жестко фиксируем 10 потоков
    )

Отпаршена 8258 страница
Отпаршена 8259 страница
Отпаршена 13760 страница
Отпаршена 8260 страница
Отпаршена 8261 страница
Отпаршена 8262 страница
Отпаршена 13761 страница
Отпаршена 8263 страница
Отпаршена 8264 страница
Отпаршена 22013 страница
Отпаршена 8265 страница
Отпаршена 8266 страница
Отпаршена 5507 страница
Отпаршена 8267 страница
Отпаршена 5508 страница
Отпаршена 22014 страница
Отпаршена 13762 страница
Отпаршена 24764 страница
Отпаршена 22015 страница
Отпаршена 22016 страница
Отпаршена 2756 страница
Отпаршена 11009 страница
Отпаршена 19262 страница
Отпаршена 5 страница
Отпаршена 19263 страница
Отпаршена 16511 страница
Отпаршена 6 страница
Отпаршена 19264 страница
Отпаршена 5509 страница
Отпаршена 7 страница
Отпаршена 5510 страница
Отпаршена 24765 страница
Отпаршена 24766 страница
Отпаршена 24767 страница
Отпаршена 13763 страница
Отпаршена 24768 страница
Отпаршена 13764 страница
Отпаршена 24769 страница
Отпаршена 8 страницаОтпаршена 24770 страница

Отпаршена 8268 страница
Отпарше